# Agent support baseline launcher
Select a Colab GPU runtime. Set `GITHUB_REPO` to the pushed repository URL. Run cells in order. Results and manifests are saved on Google Drive. Review smoke trajectories before enabling pilots. The Python package in the repository contains the experiment logic.


In [ ]:
GITHUB_REPO = ''  # e.g. https://github.com/USER/AgenticSDS.git
REVISION = 'origin/main'  # Prefer a commit SHA for a frozen pilot
DRIVE_ROOT = '/content/drive/MyDrive/agent_support'
RUN_SMOKE = True
RUN_PILOT = False  # Set True only after inspecting smoke results
assert GITHUB_REPO.startswith('https://github.com/'), 'Set GITHUB_REPO first'


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, os, urllib.request
os.environ['PYTHONHASHSEED'] = '0'  # Applies to child Python processes below
drive.mount('/content/drive')
repo = Path('/content/agent_support_repo')
if not repo.exists():
    subprocess.run(['git', 'clone', GITHUB_REPO, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', REVISION], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(repo / 'requirements.txt')], check=True)
os.chdir(repo)
config_file = Path('/content/alfworld_config.yaml')
if not config_file.exists():
    urllib.request.urlretrieve('https://raw.githubusercontent.com/alfworld/alfworld/master/configs/base_config.yaml', config_file)
data_root = Path.home() / '.cache' / 'alfworld'
if not (data_root / 'json_2.1.1').exists():
    subprocess.run(['alfworld-download'], check=True)
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
from pathlib import Path
import subprocess, sys, shutil
root = Path(DRIVE_ROOT)
for benchmark in ('alfworld',):
    folder = root / 'manifests' / benchmark
    if not (folder / 'development.json').exists() or not (folder / 'heldout.json').exists():
        command = [sys.executable, '-m', 'agent_support.manifests', benchmark, '--dev-count', '20', '--heldout-count', '100', '--output-dir', str(folder)]
        command += ['--config', str(config_file), '--split', 'eval_in_distribution']
        subprocess.run(command, check=True)
textcraft_folder = root / 'manifests' / 'textcraft'
textcraft_folder.mkdir(parents=True, exist_ok=True)
for name in ('development.json', 'heldout.json'):
    source = repo / 'manifests' / 'textcraft' / name
    target = textcraft_folder / name
    if target.exists() and target.read_bytes() != source.read_bytes():
        raise RuntimeError(f'Drive manifest differs from repository: {target}')
    if not target.exists():
        shutil.copy2(source, target)
print('Manifests ready under', root / 'manifests')


In [ ]:
if RUN_SMOKE:
    for benchmark in ('alfworld', 'textcraft'):
        command = [sys.executable, '-m', 'agent_support.runner', '--config', str(repo / 'configs' / f'{benchmark}_baseline.json'), '--manifest', str(root / 'manifests' / benchmark / 'development.json'), '--output-root', str(root / 'results'), '--limit', '3']
        subprocess.run(command, check=True)
print('Review each smoke run in Drive: run.json, summary.json, and episodes/*.json')


In [ ]:
if RUN_PILOT:
    for benchmark in ('alfworld', 'textcraft'):
        command = [sys.executable, '-m', 'agent_support.runner', '--config', str(repo / 'configs' / f'{benchmark}_baseline.json'), '--manifest', str(root / 'manifests' / benchmark / 'development.json'), '--output-root', str(root / 'results')]
        subprocess.run(command, check=True)
print('Pilot runs complete only when run.json reports status=complete.')
